In [1]:
!pip install --upgrade pip

ERROR: To modify pip, please run the following command:
C:\Users\Pongo\Desktop\Belajar\ai-models\helmet-detection\venv\Scripts\python.exe -m pip install --upgrade pip

[notice] A new release of pip available: 22.2.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached pip-26.1.2-py3-none-any.whl (1.8 MB)


In [2]:
!pip install torch==2.0.1 torchvision==0.15.2 torchaudio==2.0.2 --index-url https://download.pytorch.org/whl/cu118

Looking in indexes: https://download.pytorch.org/whl/cu118


[notice] A new release of pip available: 22.2.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip



  Using cached https://download-r2.pytorch.org/whl/cu118/torch-2.0.1%2Bcu118-cp310-cp310-win_amd64.whl (2619.1 MB)
  Using cached https://download-r2.pytorch.org/whl/cu118/torchvision-0.15.2%2Bcu118-cp310-cp310-win_amd64.whl (4.9 MB)
  Using cached https://download-r2.pytorch.org/whl/cu118/torchaudio-2.0.2%2Bcu118-cp310-cp310-win_amd64.whl (2.5 MB)
  Attempting uninstall: torch
    Found existing installation: torch 2.12.0
    Uninstalling torch-2.12.0:
      Successfully uninstalled torch-2.12.0
  Attempting uninstall: torchvision
    Found existing installation: torchvision 0.27.0
    Uninstalling torchvision-0.27.0:
      Successfully uninstalled torchvision-0.27.0


In [3]:
!pip install ultralytics opencv-python matplotlib seaborn pandas tqdm jupyter ipykernel jupyter notebook pillow streamlit scikit-learn


[notice] A new release of pip available: 22.2.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


### **Cek GPU**

In [8]:
import torch
import torch.nn as nn
import torch.optim as optim
import time
import numpy as np

In [9]:
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0))

CUDA available: True
Device: NVIDIA GeForce RTX 2050


In [10]:
# 1. Cek ketersediaan CUDA
cuda_available = torch.cuda.is_available()
print(f"CUDA Available: {cuda_available}")   # Harus True

if cuda_available:
    # 2. Jumlah GPU
    print(f"Number of GPUs: {torch.cuda.device_count()}")

    # 3. Nama GPU
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

    # 4. Kemampuan Komputasi GPU
    print(f"Compute Capability: {torch.cuda.get_device_capability(0)}")

    # 5. Versi CUDA yang digunakan oleh PyTorch
    print(f"PyTorch CUDA Version: {torch.version.cuda}")
else:
    print("PyTorch tidak mendeteksi GPU. Periksa instalasi driver dan PyTorch.")

CUDA Available: True
Number of GPUs: 1
GPU Name: NVIDIA GeForce RTX 2050
Compute Capability: (8, 6)
PyTorch CUDA Version: 11.8


In [11]:
# Buat tensor acak di GPU
x = torch.randn(1000, 1000).cuda()
y = torch.randn(1000, 1000).cuda()

# Operasi matriks perkalian
z = torch.matmul(x, y)

print("Test 1 Berhasil: Tensor berhasil diproses di GPU")
print(f"Hasil shape: {z.shape}")
print(f"Lokasi tensor: {z.device}")  # Harus 'cuda:0'

Test 1 Berhasil: Tensor berhasil diproses di GPU
Hasil shape: torch.Size([1000, 1000])
Lokasi tensor: cuda:0


In [12]:
# 1. Buat model sederhana (MLP 2 lapis)
class SimpleModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(128, 256)
        self.fc2 = nn.Linear(256, 10)
    
    def forward(self, x):
        x = torch.relu(self.fc1(x))
        return self.fc2(x)

model = SimpleModel().cuda()  # Pindahkan model ke GPU
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 2. Buat data dummy (batch size = 64)
batch_size = 64
dummy_input = torch.randn(batch_size, 128).cuda()
dummy_target = torch.randint(0, 10, (batch_size,)).cuda()

# 3. Jalankan 1 langkah training
optimizer.zero_grad()
output = model(dummy_input)
loss = criterion(output, dummy_target)
loss.backward()  # <-- Proses backpropagation di GPU
optimizer.step()

print("Test 2 Berhasil: Forward & Backward pass berjalan mulus di GPU")
print(f"Nilai Loss: {loss.item():.4f}")

Test 2 Berhasil: Forward & Backward pass berjalan mulus di GPU
Nilai Loss: 2.3525


In [13]:
# Fungsi untuk menampilkan penggunaan memori
def print_memory_usage():
    allocated = torch.cuda.memory_allocated(0) / 1024**3  # dalam GB
    reserved = torch.cuda.memory_reserved(0) / 1024**3    # dalam GB
    print(f"VRAM Terpakai (Allocated): {allocated:.2f} GB")
    print(f"VRAM Dicadangkan (Reserved): {reserved:.2f} GB")

print("Sebelum alokasi besar:")
print_memory_usage()

# Buat tensor besar untuk mengisi VRAM (misal 1 juta parameter)
big_tensor = torch.randn(10000, 10000).cuda()  # ~400 MB

print("\nSetelah alokasi tensor besar:")
print_memory_usage()

# Hapus tensor agar tidak mengganggu tes selanjutnya
del big_tensor
torch.cuda.empty_cache()
print("\nSetelah cache dibersihkan:")
print_memory_usage()

Sebelum alokasi besar:
VRAM Terpakai (Allocated): 0.03 GB
VRAM Dicadangkan (Reserved): 0.04 GB

Setelah alokasi tensor besar:
VRAM Terpakai (Allocated): 0.40 GB
VRAM Dicadangkan (Reserved): 0.41 GB

Setelah cache dibersihkan:
VRAM Terpakai (Allocated): 0.03 GB
VRAM Dicadangkan (Reserved): 0.04 GB


In [14]:
size = 5000

# --- CPU ---
start = time.time()
a_cpu = torch.randn(size, size)
b_cpu = torch.randn(size, size)
c_cpu = torch.matmul(a_cpu, b_cpu)
time_cpu = time.time() - start

# --- GPU ---
start = time.time()
a_gpu = torch.randn(size, size).cuda()
b_gpu = torch.randn(size, size).cuda()
c_gpu = torch.matmul(a_gpu, b_gpu)
torch.cuda.synchronize()  # Pastikan proses GPU selesai
time_gpu = time.time() - start

print(f"Waktu CPU: {time_cpu:.4f} detik")
print(f"Waktu GPU: {time_gpu:.4f} detik")
print(f"Percepatan GPU: {time_cpu / time_gpu:.2f}x lebih cepat")

Waktu CPU: 0.9356 detik
Waktu GPU: 0.6027 detik
Percepatan GPU: 1.55x lebih cepat


In [15]:
!python --version
!pip --version
!pip freeze > ../../requirements.txt

Python 3.10.8
pip 22.2.2 from C:\Users\Pongo\Desktop\Belajar\ai-models\helmet-detection\venv\lib\site-packages\pip (python 3.10)

